# Ionospheric electrodynamics with SWIPE

This workbook uses empirical models derived from Swarm satellite measurements to explore opolar ionospheric elecrodynamics: convection, electric field, currents, magnetic field disturbances, Joule heating. The empirical model of currents and magnetic field measurements is called AMPS. The SWIPE model combines AMPS with measurements of plasma flow to calculate electric fields and other derived quantities. 

The code below uses the Python module pyswipe to calculate and plot maps. You can also go to this webpage to explore the different quantities: https://pyswipe.uib.no/

The first cell below sets the external parameters that SWIPE depends on: Solar wind velocity (`v_sw`), interplanetary magnetic field (IMF) z component (north-south) (`b_z`), IMF y component (east-west) (`b_y`), the signed angle that Earth's magnetic dipole axis makes with a plane that is perpendicular to the Sun-Earth line (`dipole_tilt`), and the F107 index. You will vary these parameters later. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from polplot import Polarplot
from pyswipe import SWIPE

# If needed in another environment:
# %pip install pyswipe
# %pip install git+https://github.com/klaundal/polplot.git

v_sw = 400.0       # solar-wind speed [km/s]
b_y = 0.0          # IMF By, GSM [nT]
b_z = -5.0         # IMF Bz, GSM [nT]
dipole_tilt = 0.0  # dipole tilt [degrees] -- positive when the north is tilted towards the sun
f107 = 120.0       # solar radio flux [sfu]
height = 110.0     # reference height [km]

swipe = SWIPE(v_sw, b_y, b_z, dipole_tilt, f107, minlat=50.0, height=height)

the next cell calculates model values that will be plotted later

In [ ]:
vector_mlat, vector_mlt = [coordinate.ravel() for coordinate in swipe.vectorgrid]
scalar_mlat, scalar_mlt = [coordinate.ravel() for coordinate in swipe.scalargrid]

E_east, E_north = swipe.get_efield_MA(return_emphi_emlambda=True)  # [mV/m]
J_east, J_north = swipe.get_AMPS_current()                         # [A/m]
v_e1, v_e2 = swipe.get_convection_vel_MA()                         # [m/s]
v_east, v_north = v_e1, -v_e2
potential = swipe.get_potential().ravel()                          # [kV]
joule_heating = swipe.get_joule_dissipation(scalar_mlat, scalar_mlt).ravel()  # [mW/m^2]

north = vector_mlat > 0
mlat, mlt = vector_mlat[north], vector_mlt[north]
E_east, E_north = E_east[north], E_north[north]
J_east, J_north = J_east[north], J_north[north]
v_east, v_north = v_east[north], v_north[north]

north_scalar = scalar_mlat > 0
scalar_mlat, scalar_mlt = scalar_mlat[north_scalar], scalar_mlt[north_scalar]
potential, joule_heating = potential[north_scalar], joule_heating[north_scalar]

# Project J along E and along b x E. In the north, b x E = (E_north, -E_east).
E_squared = E_east**2 + E_north**2
parallel_projection = (J_east * E_east + J_north * E_north) / E_squared
perpendicular_projection = (J_east * E_north - J_north * E_east) / E_squared

J_P_east, J_P_north = parallel_projection * E_east, parallel_projection * E_north
J_H_east, J_H_north = perpendicular_projection * E_north, -perpendicular_projection * E_east

## 1. Predict the current and heating patterns

The reference case has purely southward IMF. Before looking at the current maps, use the electric field at two selected locations to predict the current directions.

Recall that 

$$\mathbf J_P \parallel \mathbf E, \qquad \mathbf J_H \parallel \mathbf b\times\mathbf E.$$

Run the next cell. For locations A and B, record the expected signs of the eastward and northward components of $\mathbf J_P$ and $\mathbf J_H$. Then predict the approximate direction of the total current. A small table or four arrows drawn on paper is sufficient.

In [ ]:
prediction_points = {"A": (70, 6), "B": (70, 18)}  # magnetic latitude [deg], magnetic local time [h]

for label, (target_mlat, target_mlt) in prediction_points.items():
    mlt_difference = np.minimum(np.abs(mlt - target_mlt), 24 - np.abs(mlt - target_mlt))
    nearest = np.argmin((mlat - target_mlat)**2 + (15 * mlt_difference)**2)
    print(f"{label}: MLAT={mlat[nearest]:.1f} deg, MLT={mlt[nearest]:.1f} h, E_east={E_east[nearest]:+.1f} mV/m, E_north={E_north[nearest]:+.1f} mV/m")

In [ ]:
figure, axis = plt.subplots(figsize=(7, 7), constrained_layout=True)
polar_axis = Polarplot(axis, minlat=50, plotgrid=False)
polar_axis.plotgrid(labels=False, color="grey", linestyle=":")
polar_axis.writeLTlabels(size=18)
polar_axis.writeLATlabels(size=18)
potential_limit = np.nanmax(np.abs(potential))
potential_plot = polar_axis.contourf(scalar_mlat, scalar_mlt, potential, levels=np.linspace(-potential_limit, potential_limit, 17), cmap="RdBu_r", extend="both", zorder=-2)
polar_axis.quiver(mlat, mlt, E_north, E_east, color="black", alpha=0.8, scale=300, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 20, "label": r"$\mathbf{E}$: $20\ \mathrm{mV\,m^{-1}}$"}, angles="xy", scale_units="xy", width=0.003)
figure.colorbar(potential_plot, ax=axis, orientation="vertical", shrink=0.7, pad=0.04, label="Electric potential [kV]")
axis.set_title(r"Electric potential and electric field $\mathbf{E}$", size=18)
plt.show()

**Heating prediction:** Based only on the electric-field pattern, where would you place the strongest currents and the strongest Joule heating $Q=\Sigma_P E^2$? State your prediction before running the overview plot in the next cell. After revealing the result, evaluate what you got right and what you missed. Why is it difficult to predict these things from only the electric field?

### Show all the maps

Run the next cell and compare the current vectors at A and B with your predictions. Identify one region where the heating map could not have been predicted from $|\mathbf E|$ alone.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 21), constrained_layout=True)
axes = axes.ravel()
polar_axes = [Polarplot(ax, minlat=50, plotgrid=False) for ax in axes]
for polar_ax in polar_axes: 
    polar_ax.plotgrid(labels=False, color = 'grey', linestyle = ':')
    polar_ax.writeLTlabels(size = 18)
polar_axes[0].writeLATlabels(size = 18)
Equiver_style = dict(angles="xy", scale_units="xy", width=0.003)
Jquiver_style = dict(angles="xy", scale_units="xy", width=0.002)

potential_limit = np.nanmax(np.abs(potential))
potential_levels = np.linspace(-potential_limit, potential_limit, 17)
potential_plots = [polar_axes[index].contourf(scalar_mlat, scalar_mlt, potential, levels=potential_levels, cmap="RdBu_r", extend="both", zorder=-2) for index in [1, 3, 4]]
joule_limit = np.nanpercentile(joule_heating, 99)
joule_plot = polar_axes[5].contourf(scalar_mlat, scalar_mlt, joule_heating, levels=np.linspace(0, joule_limit, 17), cmap="inferno", extend="max", zorder=-2)

polar_axes[0].quiver(mlat, mlt, v_north, v_east, color="black", alpha=1, scale=5000, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 500, "label": r"$\mathbf{v}$: $500\ \mathrm{m\,s^{-1}}$"}, **Jquiver_style)
polar_axes[1].quiver(mlat, mlt, E_north, E_east, color="black", alpha=0.8, scale=300, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 20, "label": r"$\mathbf{E}$: $20\ \mathrm{mV\,m^{-1}}$"}, **Equiver_style)
polar_axes[2].quiver(mlat, mlt, J_north, J_east, color="black", alpha=0.8, scale=2, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 0.1, "label": r"$\mathbf{J}$: $0.1\ \mathrm{A\,m^{-1}}$"}, **Jquiver_style)
polar_axes[3].quiver(mlat, mlt, J_P_north, J_P_east, color="black", scale=2, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 0.1, "label": r"$0.1\ \mathrm{A\,m^{-1}}$"}, **Jquiver_style)
polar_axes[4].quiver(mlat, mlt, J_H_north, J_H_east, color="black", scale=2, zorder=1, qkeyproperties={"X": 0.5, "Y": -0.05, "U": 0.1, "label": r"$0.1\ \mathrm{A\,m^{-1}}$"}, **Jquiver_style)

fig.colorbar(potential_plots[0], ax=axes[1], orientation="vertical", shrink=0.7, pad=0.04, label="Electric potential [kV]")
fig.colorbar(joule_plot, ax=axes[5], orientation="vertical", shrink=0.7, pad=0.04, label=r"Joule heating [mW m$^{-2}$]")

titles = [r"Ionospheric convection $\mathbf{v}$", r"Electric potential and electric field $\mathbf{E}$", r"Total current $\mathbf{J}$", r"Electric potential and Pedersen current $\mathbf{J}_P$", r"Electric potential and Hall current $\mathbf{J}_H$", r"Joule heating"]
for ax, title in zip(axes, titles): ax.set_title(title, size = 18)

fig.suptitle(rf"SWIPE: northern hemisphere, $B_y={b_y:g}$, $B_z={b_z:g}$ nT, $v_{{sw}}={v_sw:g}$ km s$^{{-1}}$, tilt $={dipole_tilt:g}^\circ$", fontsize=15)
plt.show()

## 2. Change one external condition

Return to the first code cell and change **one** model input at a time. Here you can also go to https://pyswipe.uib.no/ if the code in the notebook runs slowly. Compare:

- $B_z=+5$ and $-8$ nT
- $B_y=-5$ and $+5$ nT
- $v_{sw}=300$ and $600$ km/s
- dipole tilt $=-25^\circ$ and $+25^\circ$

Rerun the model and overview plot for the different values. Which change is most conspicuous: the overall strength, the shape, or the symmetry of the patterns? Describe one linked response involving at least two quantities, for example convection and potential, or Pedersen current and heating.

Restore the reference values before continuing.

## 3. Why the current does not simply follow the electric field

Use the conductance maps below together with the overview plot. Choose one region where a strong current appears to be caused mainly by a strong electric field, and another where enhanced conductance seems more important. Explain how you made the distinction.

In [ ]:
Sigma_H, Sigma_P, conductance_mask = swipe.get_conductances(scalar_mlat, scalar_mlt)  # [S]
Sigma_P = np.ma.masked_where(conductance_mask, Sigma_P)
Sigma_H = np.ma.masked_where(conductance_mask, Sigma_H)

fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
conductance_axes = [Polarplot(ax, minlat=50, plotgrid=False) for ax in axes]
for polar_ax in conductance_axes:
    polar_ax.plotgrid(labels=False, color='grey', linestyle=':')
    polar_ax.writeLTlabels(size=18)
conductance_axes[0].writeLATlabels(size=18)

pedersen_limit = np.nanpercentile(Sigma_P.compressed(), 99)
hall_limit = np.nanpercentile(Sigma_H.compressed(), 99)
pedersen_plot = conductance_axes[0].contourf(scalar_mlat, scalar_mlt, Sigma_P, levels=np.linspace(0, pedersen_limit, 17), cmap='viridis', extend='max')
hall_plot = conductance_axes[1].contourf(scalar_mlat, scalar_mlt, Sigma_H, levels=np.linspace(0, hall_limit, 17), cmap='viridis', extend='max')

fig.colorbar(pedersen_plot, ax=axes[0], orientation='vertical', shrink=0.7, pad=0.04, label=r'$\Sigma_P$ [S]')
fig.colorbar(hall_plot, ax=axes[1], orientation='vertical', shrink=0.7, pad=0.04, label=r'$\Sigma_H$ [S]')
axes[0].set_title(r'Pedersen conductance $\Sigma_P$', size=18)
axes[1].set_title(r'Hall conductance $\Sigma_H$', size=18)
fig.suptitle(rf'SWIPE: northern hemisphere, $B_y={b_y:g}$, $B_z={b_z:g}$ nT, $v_{{sw}}={v_sw:g}$ km s$^{{-1}}$, tilt $={dipole_tilt:g}^\circ$', fontsize=15)
plt.show()

## 4. Equivalent current and Hall current

Ground magnetometers constrain the divergence-free equivalent current, not the complete three-dimensional current system. Compare the Hall-current panel above with the equivalent-current function below.

Repeat the comparison for dipole tilts of $-25^\circ$ and $+25^\circ$. Under which seasonal condition does the equivalent current most closely resemble the Hall current? Is the agreement equally good on the dayside and nightside?

**Follow-up:** For the same date, what would you expect in the opposite hemisphere? 

In [ ]:
from pyamps import AMPS

amps = AMPS(v_sw, b_y, b_z, dipole_tilt, f107, minlat=50.0, height=height)
FAC = amps.get_upward_current(scalar_mlat, scalar_mlt).ravel()  # [microA/m^2], positive upward
Psi = amps.get_divergence_free_current_function(scalar_mlat, scalar_mlt).ravel()  # [kA]

fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
current_axes = [Polarplot(ax, minlat=50, plotgrid=False) for ax in axes]
for current_axis in current_axes:
    current_axis.plotgrid(labels=False, color='grey', linestyle=':')
    current_axis.writeLTlabels(size=18)
current_axes[0].writeLATlabels(size=18)

fac_limit = np.nanpercentile(np.abs(FAC), 99)
fac_levels = np.linspace(-fac_limit, fac_limit, 22)
fac_plot = current_axes[0].contourf(scalar_mlat, scalar_mlt, FAC, levels=fac_levels, cmap='bwr', extend='both', zorder=-2)
current_axes[0].quiver(mlat, mlt, J_north, J_east, color='black', alpha=0.8, scale=2, zorder=1, qkeyproperties={'X': 0.5, 'Y': -0.05, 'U': 0.1, 'label': r'$\mathbf{J}$: $0.1\ \mathrm{A\,m^{-1}}$'}, **Jquiver_style)

psi_limit = np.nanmax(np.abs(Psi))
psi_levels = np.linspace(-psi_limit, psi_limit, 22)
psi_plot = current_axes[1].contour(scalar_mlat, scalar_mlt, Psi, levels=psi_levels, colors = 'black', extend='both', zorder=-2)
#current_axes[1].contour(scalar_mlat, scalar_mlt, Psi, levels=psi_levels, colors='black', linewidths=0.4, alpha=0.5)

fig.colorbar(fac_plot, ax=axes[0], orientation='vertical', shrink=0.7, pad=0.04, label=r'Upward FAC [$\mu$A m$^{-2}$]')
#fig.colorbar(psi_plot, ax=axes[1], orientation='vertical', shrink=0.7, pad=0.04, label=r'Equivalent-current function $\Psi$ [kA]')
axes[0].set_title(r'Total horizontal current $\mathbf{J}$ and field-aligned current', size=18)
axes[1].set_title(r'Equivalent-current function $\Psi$', size=18)
fig.suptitle(rf'SWIPE/AMPS: northern hemisphere, $B_y={b_y:g}$, $B_z={b_z:g}$ nT, $v_{{sw}}={v_sw:g}$ km s$^{{-1}}$, tilt $={dipole_tilt:g}^\circ$', fontsize=15)
plt.show()

## 5. A real day at Qeqertarsuaq

The files used below are already stored in the repository. GDH is the magnetic observatory at Qeqertarsuaq, Greenland. The observed disturbance is projected onto the local IGRF horizontal-field direction,

$$\Delta H=\Delta B_N\cos D+\Delta B_E\sin D,$$

and the solar-wind inputs to AMPS are smoothed with a centered rolling 20-minute mean.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import pandas as pd
import ppigrf

from pyamps import get_B_ground

repository = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_directory = repository / "data" / "gdh_20250526"
ground = pd.read_csv(data_directory / "gdh_magnetic_residual_20250526_1min.csv", parse_dates=["time_utc"], index_col="time_utc")
drivers = pd.read_csv(data_directory / "swipe_drivers_20250526_1min.csv", parse_dates=["time_utc"], index_col="time_utc")

ground = ground[["B_N_res_CHAOS_nT", "B_E_res_CHAOS_nT"]].resample("5min").mean()
solar_wind = ["By_GSM_nT", "Bz_GSM_nT", "speed_km_s"]
drivers[solar_wind] = drivers[solar_wind].interpolate(method="time", limit=30, limit_area="inside")
drivers[solar_wind] = drivers[solar_wind].rolling("20min", center=True, min_periods=10).mean()
drivers = drivers.resample("5min").mean(numeric_only=True)
required = ["By_GSM_nT", "Bz_GSM_nT", "speed_km_s", "dipole_tilt_deg", "F107_sfu", "GDH_QD_latitude_deg", "GDH_MLT_hours"]
drivers = drivers.loc[drivers[required].notna().all(axis=1)]
ground = ground.reindex(drivers.index)

B_qd_east, B_qd_north, B_up = get_B_ground(qdlat=drivers["GDH_QD_latitude_deg"].to_numpy(), mlt=drivers["GDH_MLT_hours"].to_numpy(), height=0, v=drivers["speed_km_s"].to_numpy(), By=drivers["By_GSM_nT"].to_numpy(), Bz=drivers["Bz_GSM_nT"].to_numpy(), tilt=drivers["dipole_tilt_deg"].to_numpy(), f107=drivers["F107_sfu"].to_numpy())

igrf_east, igrf_north, igrf_up = ppigrf.igrf(306.47, 69.25, 0, pd.Timestamp("2025-05-26T12:00:00").to_pydatetime())
declination = np.arctan2(float(igrf_east[0]), float(igrf_north[0]))
observed_H = ground["B_N_res_CHAOS_nT"] * np.cos(declination) + ground["B_E_res_CHAOS_nT"] * np.sin(declination)
observed_H = observed_H - observed_H.median()
modeled_H = B_qd_north - np.nanmedian(B_qd_north)

figure, axis = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
axis.plot(ground.index, observed_H, color="black", linewidth=1.8, label=r"GDH $\Delta H$")
axis.plot(drivers.index, modeled_H, color="tab:red", linewidth=1.8, label=r"AMPS $\Delta H$")
axis.axhline(0, color="0.5", linewidth=0.8)
axis.set_xlim(pd.Timestamp("2025-05-26", tz="UTC"), pd.Timestamp("2025-05-27", tz="UTC"))
axis.xaxis.set_major_locator(mdates.HourLocator(interval=3))
axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axis.set_xlabel("UTC on 26 May 2025")
axis.set_ylabel(r"$H$ disturbance [nT]")
axis.legend(loc="upper left")
axis.set_title("GDH observations and AMPS prediction")
plt.show()

Identify one interval where the observed and modeled variations agree reasonably well, and the interval containing their largest disagreement. What does the comparison suggest that AMPS represents well, and what kind of behaviour does it miss?



## 6. What would you measure next?

### What can actually be inferred?

Suppose you receive only the GDH $\Delta H$ record, without AMPS or solar-wind data. What could you reasonably conclude about the ionospheric current above Greenland? What could you **not** determine from this measurement alone? Distinguish evidence for a changing equivalent current or electrojet from claims about the full horizontal current, convection, or field-aligned currents.

### Choose one additional observation

You may add one measurement to the GDH magnetometer:

- plasma convection from a radar,
- auroral emission from an imager, or
- field-aligned current from a satellite.

Which would you choose to investigate the large negative bay near 22 UTC? Explain what ambiguity your choice would help resolve.